# Introduction to SQL with Formula 1 Data

**Learning Objective:** By the end of this notebook, you will be able to write SQL queries to explore, filter, sort, count, and join data — all using real Formula 1 racing records.

**Economics Concept:** We use data to answer questions. In economics, we ask things like "who wins the most?" or "which team dominates in which era?" SQL (Structured Query Language) is the tool that lets us ask those questions against a database.

---

## What is Formula 1?

Formula 1 is the highest class of international single-seater auto racing. Races are called **Grand Prix** (French for "big prize"). Each year is called a **season**. Drivers earn **points** for finishing well. The driver with the most points at the end of the season wins the **World Drivers' Championship**.

Teams that build the cars are called **constructors**. Ferrari, Mercedes, and Red Bull are famous constructors. The team that earns the most points wins the **Constructors' Championship**.

Our database covers every F1 race from **1950 to 2026** — over 1,000 races!



`%load_ext sql` activates the SQL magic so that cells starting with `%sql` or `%%sql` will run SQL queries instead of Python.

In [ ]:
import pandas as pd
import numpy as np
import sqlalchemy
from pathlib import Path
%load_ext sql

##Connect to the Database

A **database** is like a collection of spreadsheets, all stored in one file. Our F1 database file is called `f1db.db`.

We use `sqlalchemy.create_engine` to open a connection to it. Think of this like opening a file — before you can read it, you have to open it first.

In [ ]:
# Connect to the F1 database file
f1_engine = sqlalchemy.create_engine('sqlite:///f1db.db')
f1_connection = f1_engine.connect()

Now we tell the `%sql` magic to use this connection.

In [ ]:
%sql f1_engine

## Step 4: What Tables Are in the Database?

A database contains many **tables**. Each table is like a spreadsheet with rows and columns.

The query below asks SQLite (the type of database we are using) to list all the tables. `sqlite_master` is a special internal table that stores information about the database structure.

In [ ]:
%%sql
SELECT name FROM sqlite_master WHERE type = 'table';

There are many tables! The most important ones for us are:

| Table | What it contains |
|---|---|
| `driver` | Every driver who has ever competed in F1 |
| `constructor` | Every team (car builder) |
| `circuit` | Every racetrack |
| `race` | Every race, including date and location |
| `race_data` | Detailed results: qualifying times, race results, pit stops |
| `season_driver_standing` | Driver championship standings at end of each season |
| `grand_prix` | Information about each Grand Prix event |

## Step 5: Meet the Drivers

Let's start with the `driver` table. The SQL command `SELECT * FROM driver` means:
- `SELECT` — choose some columns
- `*` — the asterisk means "all columns"
- `FROM driver` — from the table called `driver`

We add `LIMIT 10` so we only see the first 10 rows instead of all 900+ drivers.

In [ ]:
%sql SELECT * FROM driver LIMIT 10;

That is a lot of columns! Let's look at just the column names by using `PRAGMA table_info`. This tells us the **schema** (structure) of the table — what each column is called and what type of data it holds.

In [ ]:
%sql PRAGMA table_info(driver);

Now let's select only the columns that are most interesting to us — the driver's name, nationality, and career stats.

In [ ]:
%%sql
SELECT name, nationality_country_id, total_race_wins, total_championship_wins, total_pole_positions
FROM driver
LIMIT 15;

## Step 6: How Many Drivers Are in the Database?

`COUNT(*)` counts how many rows are in a table. This is like asking "how many records do we have?"

`AS total_drivers` gives the result a friendly column name. Without it, the column would just be called `COUNT(*)`.

In [ ]:
%sql SELECT COUNT(*) AS total_drivers FROM driver;

Over 900 drivers have competed in Formula 1 since 1950! Now let's count how many races have been held.

In [ ]:
%sql SELECT COUNT(*) AS total_races FROM race;

## Step 7: Filter with WHERE

The `WHERE` clause lets us filter rows. It is like asking: "show me only the rows where a condition is true."

Let's find all drivers who have won at least one World Championship. The column `total_championship_wins` tells us how many titles a driver has won.

The `>` symbol means "greater than."

In [ ]:
%%sql
SELECT name, nationality_country_id, total_championship_wins, total_race_wins
FROM driver
WHERE total_championship_wins > 0;

Let's also filter by nationality. The `=` symbol checks for an exact match. Here we look for all drivers from Brazil.

In [ ]:
%%sql
SELECT name, total_race_wins, total_championship_wins, total_pole_positions
FROM driver
WHERE nationality_country_id = 'brazil';

Brazil has produced some legendary drivers! Ayrton Senna, one of the most celebrated drivers in history, is in this list.

We can also combine conditions using `AND`. Let's find all British drivers who have won at least 20 races.

In [ ]:
%%sql
SELECT name, total_race_wins, total_championship_wins
FROM driver
WHERE nationality_country_id = 'united-kingdom'
AND total_race_wins >= 20;

## Step 8: Sort Results with ORDER BY

`ORDER BY` sorts the results. By default it sorts from smallest to largest (`ASC` = ascending). To sort from largest to smallest, we add `DESC` (descending).

Let's find the top 10 drivers with the most race wins of all time.

In [ ]:
%%sql
SELECT name, nationality_country_id, total_race_wins, total_championship_wins
FROM driver
ORDER BY total_race_wins DESC
LIMIT 10;

Lewis Hamilton leads with over 100 wins! Michael Schumacher held the record for decades before Hamilton broke it.

Now let's see who has the most **pole positions**. A pole position means you qualified fastest and start the race in first place.

In [ ]:
%%sql
SELECT name, total_pole_positions, total_race_wins, total_championship_wins
FROM driver
ORDER BY total_pole_positions DESC
LIMIT 10;

## Step 9: Explore the Circuits

A **circuit** is a racetrack. Each circuit has a name, location, length (in kilometers), and a count of how many times it has hosted a Grand Prix.

Let's look at all the circuits in the database.

In [ ]:
%%sql
SELECT name, place_name, country_id, length, turns, total_races_held
FROM circuit
ORDER BY total_races_held DESC
LIMIT 15;

**Monza** in Italy has hosted the most races — it has been part of the F1 calendar since the very first season in 1950. **Monaco** is the most famous street circuit: drivers race through the narrow roads of the city, making it one of the most challenging races in the world.

Let's look at the **longest circuits** — the ones with the most kilometers per lap.

In [ ]:
%%sql
SELECT name, country_id, length, turns
FROM circuit
WHERE length IS NOT NULL
ORDER BY length DESC
LIMIT 10;

## Step 10: Explore the Races

The `race` table has one row for every race ever held. Let's look at the most recent races.

In [ ]:
%%sql
SELECT year, round, official_name, circuit_id, date
FROM race
ORDER BY date DESC
LIMIT 10;

Now let's look at the very first F1 races from 1950.

In [ ]:
%%sql
SELECT year, round, official_name, circuit_id, date
FROM race
ORDER BY date ASC
LIMIT 10;

## Step 11: GROUP BY — Counting by Category

`GROUP BY` lets us split the data into groups and then count or sum within each group. 

Think of it like sorting a pile of papers by color, then counting how many papers are in each color pile.

Let's count how many races have been held each decade. The `year / 10 * 10` trick rounds the year down to the nearest decade (for example, 1987 becomes 1980).

In [ ]:
%%sql
SELECT (year / 10 * 10) AS decade, COUNT(*) AS races_that_decade
FROM race
GROUP BY decade
ORDER BY decade ASC;

Notice how the number of races per season has grown over time. In the 1950s there were only a handful of races per season. Today there are over 20 races per season.

Now let's count how many drivers have raced from each country. This uses the `driver` table.

In [ ]:
%%sql
SELECT nationality_country_id AS country, COUNT(*) AS number_of_drivers
FROM driver
GROUP BY nationality_country_id
ORDER BY number_of_drivers DESC
LIMIT 15;

The United Kingdom and the United States have historically produced the most F1 drivers. The UK in particular has been a center of F1 engineering and racing since the sport began.

Now let's count wins by constructor. Which team has the most wins ever?

In [ ]:
%%sql
SELECT name, country_id, total_race_wins, total_championship_wins
FROM constructor
ORDER BY total_race_wins DESC
LIMIT 10;

**Ferrari** leads all teams with the most wins. They have been competing since the very first F1 season in 1950. McLaren and Mercedes are close behind. Red Bull, despite being a relatively new team, has rapidly climbed the rankings.

## Step 12: JOIN — Connecting Two Tables

A `JOIN` links two tables together using a shared column. This is one of the most powerful SQL features.

For example, the `race` table has a `circuit_id` column (like `'monaco'`). The `circuit` table has an `id` column that matches. We can join them to get the full circuit name alongside each race.

The syntax is:
```sql
FROM table1
JOIN table2 ON table1.shared_column = table2.shared_column
```

Let's join `race` and `circuit` to see the full circuit name for each recent race.

In [ ]:
%%sql
SELECT race.year, race.round, race.official_name,
       circuit.full_name AS circuit_full_name,
       circuit.place_name,
       circuit.country_id
FROM race
JOIN circuit ON race.circuit_id = circuit.id
ORDER BY race.date DESC
LIMIT 10;

Now let's do a more interesting join. The `race_data` table stores race results. It has a `driver_id` column (like `'lewis-hamilton'`). The `driver` table has an `id` column. We can join them to get the driver's readable name alongside each result.

Let's find the winner of each race in the 2023 season. We filter `race_data` to only `RACE_RESULT` type rows, and to `position_number = 1` (first place).

In [ ]:
%%sql
SELECT race.round, race.official_name, driver.name AS race_winner, constructor.name AS team
FROM race_data
JOIN race ON race_data.race_id = race.id
JOIN driver ON race_data.driver_id = driver.id
JOIN constructor ON race_data.constructor_id = constructor.id
WHERE race.year = 2023
AND race_data.type = 'RACE_RESULT'
AND race_data.position_number = 1
ORDER BY race.round ASC;

Max Verstappen dominated the 2023 season, winning the vast majority of races for Red Bull. Let's count exactly how many races each driver won that year.

In [ ]:
%%sql
SELECT driver.name AS driver_name, constructor.name AS team, COUNT(*) AS wins_in_2023
FROM race_data
JOIN race ON race_data.race_id = race.id
JOIN driver ON race_data.driver_id = driver.id
JOIN constructor ON race_data.constructor_id = constructor.id
WHERE race.year = 2023
AND race_data.type = 'RACE_RESULT'
AND race_data.position_number = 1
GROUP BY driver.name, constructor.name
ORDER BY wins_in_2023 DESC;

## Step 13: Championship Standings

The `season_driver_standing` table records the final championship standings for each season. Let's look at who has won the Drivers' Championship each year.

We filter `championship_won = 1` to get only the champion.

In [ ]:
%%sql
SELECT season_driver_standing.year,
       driver.name AS world_champion,
       driver.nationality_country_id AS country,
       season_driver_standing.points
FROM season_driver_standing
JOIN driver ON season_driver_standing.driver_id = driver.id
WHERE season_driver_standing.championship_won = 1
ORDER BY season_driver_standing.year DESC
LIMIT 20;

Let's see which driver has won the championship the most times by counting their title wins.

In [ ]:
%%sql
SELECT driver.name AS driver_name,
       driver.nationality_country_id AS country,
       COUNT(*) AS championships_won
FROM season_driver_standing
JOIN driver ON season_driver_standing.driver_id = driver.id
WHERE season_driver_standing.championship_won = 1
GROUP BY driver.name, driver.nationality_country_id
ORDER BY championships_won DESC
LIMIT 10;

Lewis Hamilton and Michael Schumacher share the record with 7 championships each. Juan Manuel Fangio, who raced in the 1950s, won 5 titles and is widely considered one of the greatest drivers ever.

Now let's look at the Constructors' Championship — which team has dominated the most?

In [ ]:
%%sql
SELECT constructor.name AS team,
       constructor.country_id AS country,
       COUNT(*) AS constructors_titles_won
FROM season_constructor_standing
JOIN constructor ON season_constructor_standing.constructor_id = constructor.id
WHERE season_constructor_standing.championship_won = 1
GROUP BY constructor.name, constructor.country_id
ORDER BY constructors_titles_won DESC;

**Ferrari** leads the all-time Constructors' Championship count. **Williams**, which is often underestimated today, had a remarkable run in the 1980s and 1990s. **Mercedes** dominated the sport from 2014 to 2021, winning 8 consecutive titles.

## Step 14: Save SQL Results to a Pandas DataFrame

So far we have been reading results directly in the notebook. But sometimes we want to save the results so we can work with them further in Python — for example, to make a chart.

We use the `<<` operator to save the SQL result to a variable, and then call `.DataFrame()` to convert it to a pandas DataFrame.

Let's save the top 10 drivers by wins.

In [ ]:
%%sql top_drivers_by_wins <<
SELECT name AS driver_name, nationality_country_id AS country,
       total_race_wins, total_championship_wins, total_pole_positions
FROM driver
ORDER BY total_race_wins DESC
LIMIT 10;

Now we convert the SQL result to a pandas DataFrame and display it.

In [ ]:
top_drivers_dataframe = top_drivers_by_wins.DataFrame()
top_drivers_dataframe

Now that we have a pandas DataFrame, we can create a bar chart showing career race wins for the top 10 drivers.

In [ ]:
import matplotlib.pyplot as plt   # for creating charts

plt.figure(figsize=(10, 6))
plt.bar(top_drivers_dataframe['driver_name'], top_drivers_dataframe['total_race_wins'], color='red')
plt.xlabel('Driver')
plt.ylabel('Career Race Wins')
plt.title('Top 10 Formula 1 Drivers by Career Race Wins')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

Lewis Hamilton's bar towers above the rest. His 105 wins are the result of both exceptional talent and the advantage of driving for dominant teams — first McLaren, then Mercedes during their dominant era from 2014 to 2020.

## Step 15: Championship Points Over Time

Let's look at a specific driver's championship points across their career. We will look at Max Verstappen, the current dominant driver.

We join `season_driver` (which has per-season totals) with `driver` to get the name.

In [ ]:
%%sql verstappen_seasons <<
SELECT season_driver.year, season_driver.total_race_wins,
       season_driver.total_podiums, season_driver.total_points,
       season_driver.position_number AS championship_position
FROM season_driver
JOIN driver ON season_driver.driver_id = driver.id
WHERE driver.name = 'Max Verstappen'
ORDER BY season_driver.year ASC;

Now we convert the query result to a pandas DataFrame and display it.

In [ ]:
verstappen_career_data = verstappen_seasons.DataFrame()
verstappen_career_data

We can see Max Verstappen's career trajectory — from his debut as a teenager in 2015, through his first championship in 2021, to his dominance from 2022 onward. His points totals grew dramatically as Red Bull built a stronger car around him.

## Step 16: Advanced Query — Dominant Eras

Let's identify which team won the Constructors' Championship in each year. This shows us how dominance in F1 shifts between different eras.

We use `season_constructor_standing` joined with `constructor`. We filter to `championship_won = 1` to get only the champion each year.

In [ ]:
%%sql
SELECT season_constructor_standing.year,
       constructor.name AS constructors_champion
FROM season_constructor_standing
JOIN constructor ON season_constructor_standing.constructor_id = constructor.id
WHERE season_constructor_standing.championship_won = 1
ORDER BY season_constructor_standing.year ASC;

You can clearly see dominant eras in F1 history:
- **Ferrari** dominated the 1950s and early 2000s
- **Lotus** was strong in the 1960s
- **McLaren** and **Williams** traded titles through the 1980s and 1990s
- **Mercedes** won 8 consecutive titles from 2014 to 2021
- **Red Bull** has dominated since 2022

These dominant periods are often driven by technical rule changes that favor one team's design philosophy — very similar to how firms gain competitive advantages in economics.

## Step 17: Explore Race Results in Detail

The `race_data` table is the most detailed table in the database. Each row can represent a race result, a qualifying time, a pit stop, and more. The `type` column tells you what kind of record it is.

Let's look at the full race results for one specific race: the 2024 Monaco Grand Prix. First we need to find the race ID.

In [ ]:
%%sql
SELECT id, year, round, official_name, date
FROM race
WHERE grand_prix_id = 'monaco'
AND year = 2024;

Now we use that race ID to look up the race results and join the driver and constructor tables to get readable names.

In [ ]:
%%sql
SELECT race_data.position_number AS finishing_position,
       race_data.driver_number,
       driver.name AS driver_name,
       constructor.name AS team,
       race_data.race_laps AS laps_completed,
       race_data.race_time AS race_time,
       race_data.race_gap AS gap_to_leader,
       race_data.race_points AS points_scored
FROM race_data
JOIN race ON race_data.race_id = race.id
JOIN driver ON race_data.driver_id = driver.id
JOIN constructor ON race_data.constructor_id = constructor.id
WHERE race.grand_prix_id = 'monaco'
AND race.year = 2024
AND race_data.type = 'RACE_RESULT'
ORDER BY race_data.position_display_order ASC;

## Step 18: HAVING — Filter Groups

`HAVING` works like `WHERE`, but it filters **after** the `GROUP BY` step. We use it to filter based on aggregated values.

For example: find all circuits that have hosted more than 30 races.

In [ ]:
%%sql
SELECT name AS circuit_name, country_id, total_races_held
FROM circuit
WHERE total_races_held > 30
ORDER BY total_races_held DESC;

Now let's use `HAVING` in a GROUP BY context. Let's find all constructors who have won at least 5 races in a single season. We look at the `season_constructor` table.

In [ ]:
%%sql
SELECT constructor.name AS team_name, season_constructor.year, season_constructor.total_race_wins
FROM season_constructor
JOIN constructor ON season_constructor.constructor_id = constructor.id
WHERE season_constructor.total_race_wins >= 15
ORDER BY season_constructor.total_race_wins DESC
LIMIT 15;

The most dominant single-season performances in history stand out clearly here. Red Bull's 2023 season — with 21 wins from 22 races — was the most dominant season in F1 history.

## Step 19: Subqueries — Queries Inside Queries

A **subquery** is a SQL query nested inside another query. It lets us use the result of one query as a filter or source for another.

For example: find all drivers who have ever raced at the Monaco circuit. We need to:
1. Find the IDs of all Monaco races (inner query)
2. Find all drivers who appear in the results for those races (outer query)

This uses `IN`, which checks whether a value is in a list.

In [ ]:
%%sql
SELECT DISTINCT driver.name AS driver_name, driver.nationality_country_id AS country
FROM race_data
JOIN driver ON race_data.driver_id = driver.id
WHERE race_data.type = 'RACE_RESULT'
AND race_data.race_id IN (
    SELECT id FROM race WHERE grand_prix_id = 'monaco'
)
ORDER BY driver.name ASC
LIMIT 20;

Now let's answer a trickier question: which drivers have **won** at Monaco? We add an extra filter for `position_number = 1`.

In [ ]:
%%sql
SELECT driver.name AS driver_name,
       driver.nationality_country_id AS country,
       COUNT(*) AS monaco_wins
FROM race_data
JOIN driver ON race_data.driver_id = driver.id
WHERE race_data.type = 'RACE_RESULT'
AND race_data.position_number = 1
AND race_data.race_id IN (
    SELECT id FROM race WHERE grand_prix_id = 'monaco'
)
GROUP BY driver.name, driver.nationality_country_id
ORDER BY monaco_wins DESC
LIMIT 10;

Ayrton Senna holds the record for the most Monaco wins — 6 victories at the most prestigious street circuit in the world. His mastery of Monaco is considered one of the greatest skills in motorsport history.

## Summary: What You Learned

Congratulations! You have explored over 70 years of Formula 1 data using SQL. Here is a recap of the SQL tools you used:

| SQL Command | What it does |
|---|---|
| `SELECT ... FROM ...` | Choose columns from a table |
| `LIMIT` | Show only a certain number of rows |
| `WHERE` | Filter rows by a condition |
| `AND`, `OR` | Combine conditions |
| `ORDER BY ... DESC/ASC` | Sort the results |
| `COUNT(*)` | Count the number of rows |
| `GROUP BY` | Split data into groups before counting |
| `HAVING` | Filter after grouping |
| `JOIN ... ON` | Combine two tables using a shared column |
| `IN (subquery)` | Filter using the result of another query |
| `DISTINCT` | Return only unique values |

**Economics Takeaway:** SQL is how economists access large datasets stored in relational databases. The same logic you used here — filtering, counting, grouping, joining — applies directly when you work with economic data such as labor surveys, trade records, or firm-level data. Every query is a question. Learning to write good questions is the first step toward good data analysis.

**F1 Takeaway:** Dominance in Formula 1 comes in cycles. Technical rule changes, engineering investment, and driver talent all interact to determine who wins. The data shows that no single team or driver dominates forever — and that is what makes the sport exciting.